In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
def _as_array(x, n, dtype=float, name="parameter"):
    """
    Convert scalar or sequence input to an array of length n.
    """
    if np.isscalar(x):
        return np.full(n, x, dtype=dtype)

    arr = np.asarray(x, dtype=dtype)
    if arr.shape[0] != n:
        raise ValueError(f"{name} must be either scalar or length {n}.")
    return arr

In [ ]:
def _sample_noise(rng, shape, noise_std=1.0, noise_distribution="gaussian", df=5):
    """
    Generate background noise.
    """
    if noise_distribution == "gaussian":
        return rng.normal(0.0, noise_std, size=shape)

    if noise_distribution == "laplace":
        return rng.laplace(0.0, noise_std / np.sqrt(2), size=shape)

    if noise_distribution == "student_t":
        if df <= 2:
            raise ValueError("Student-t noise requires df > 2 for finite variance.")
        raw = rng.standard_t(df=df, size=shape)
        return raw * noise_std / np.sqrt(df / (df - 2))

    raise ValueError("noise_distribution must be 'gaussian', 'laplace', or 'student_t'.")

In [4]:
def _make_supports(
    total_size,
    n_clusters,
    size_fraction=0.1,
    overlap_ratio=0.0,
    rng=None,
    placement="diagonal",
    size_jitter=0.0,
):
    """
    Construct row or column supports for biclusters.

    Parameters
    ----------
    total_size : int
        Number of rows or columns.
    n_clusters : int
        Number of biclusters.
    size_fraction : float or sequence of floats
        Fraction of rows/columns assigned to each bicluster.
    overlap_ratio : float
        Desired overlap area ratio between consecutive biclusters.
    placement : {'diagonal', 'random'}
        'diagonal' gives contiguous supports before shuffling.
        'random' gives randomly placed supports.
    size_jitter : float
        Random multiplicative jitter in cluster sizes.
        Example: 0.2 means sizes are multiplied by Uniform(0.8, 1.2).

    Returns
    -------
    indicators : ndarray, shape (total_size, n_clusters)
        Binary membership matrix.
    supports : list of ndarray
        List of index arrays, one per bicluster.
    """
    if rng is None:
        rng = np.random.default_rng()

    if not (0 <= overlap_ratio < 1):
        raise ValueError("overlap_ratio must be in [0, 1).")

    fractions = _as_array(size_fraction, n_clusters, dtype=float, name="size_fraction")

    if np.any(fractions <= 0) or np.any(fractions > 1):
        raise ValueError("All size fractions must be in (0, 1].")

    sizes = np.maximum(1, np.round(fractions * total_size).astype(int))

    if size_jitter > 0:
        jitter = rng.uniform(1 - size_jitter, 1 + size_jitter, size=n_clusters)
        sizes = np.maximum(1, np.round(sizes * jitter).astype(int))

    if np.any(sizes > total_size):
        raise ValueError("A requested cluster size is larger than the dimension.")

    indicators = np.zeros((total_size, n_clusters), dtype=np.int8)
    supports = []

    if placement == "diagonal":
        start = 0

        for k in range(n_clusters):
            size = sizes[k]

            if k > 0:
                previous_size = sizes[k - 1]
                overlap = int(round(overlap_ratio * min(previous_size, size)))
                start = supports[-1][-1] + 1 - overlap

            end = start + size

            if end > total_size:
                raise ValueError(
                    "The requested diagonal supports do not fit. "
                    "Reduce n_clusters, size_fraction, overlap_ratio, or size_jitter."
                )

            idx = np.arange(start, end)
            supports.append(idx)
            indicators[idx, k] = 1

    elif placement == "random":
        used = set()

        for k in range(n_clusters):
            size = sizes[k]

            if k == 0:
                idx = rng.choice(total_size, size=size, replace=False)

            else:
                previous = supports[-1]
                overlap = int(round(overlap_ratio * min(len(previous), size)))

                overlap_idx = (
                    rng.choice(previous, size=overlap, replace=False)
                    if overlap > 0
                    else np.array([], dtype=int)
                )

                new_needed = size - overlap
                forbidden = set(overlap_idx.tolist())

                # Prefer new indices not already used, so overlap is mainly controlled.
                candidates = np.array(
                    [i for i in range(total_size) if i not in used and i not in forbidden]
                )

                if len(candidates) < new_needed:
                    # Fall back to any indices not already in the current overlap set.
                    candidates = np.array(
                        [i for i in range(total_size) if i not in forbidden]
                    )

                if len(candidates) < new_needed:
                    raise ValueError("Not enough indices available to create support.")

                new_idx = rng.choice(candidates, size=new_needed, replace=False)
                idx = np.concatenate([overlap_idx, new_idx])

            idx = np.sort(idx)
            supports.append(idx)
            indicators[idx, k] = 1
            used.update(idx.tolist())

    else:
        raise ValueError("placement must be either 'diagonal' or 'random'.")

    return indicators, supports

In [5]:
def _shuffle_outputs(data, row_indicators, col_indicators, rng):
    """
    Shuffle rows and columns and return shuffled data and shuffled truth masks.
    """
    row_perm = rng.permutation(data.shape[0])
    col_perm = rng.permutation(data.shape[1])

    shuffled_data = data[row_perm][:, col_perm]
    shuffled_row_indicators = row_indicators[row_perm, :]
    shuffled_col_indicators = col_indicators[col_perm, :]

    return (
        shuffled_data,
        shuffled_row_indicators,
        shuffled_col_indicators,
        row_perm,
        col_perm,
    )

In [6]:
def generate_additive_biclusters(
    n_clusters=3,
    total_rows=1000,
    total_cols=100,
    row_fraction=0.1,
    col_fraction=0.1,
    contrast_level=3.0,
    row_overlap_ratio=0.15,
    col_overlap_ratio=0.15,
    background_noise_std=1.0,
    row_effect_std=0.2,
    col_effect_std=0.2,
    within_block_noise_std=0.1,
    size_jitter=0.0,
    placement="diagonal",
    noise_distribution="gaussian",
    noise_df=5,
    signed_blocks=False,
    cell_signal_density=1.0,
    seed=42,
    dtype=np.float64,
):
    """
    Generate synthetic biclustering data using an additive Plaid-style model.

    Main controlled factors
    -----------------------
    n_clusters : number of biclusters.
    row_fraction : row support size as fraction of total rows.
    col_fraction : column support size as fraction of total columns.
    contrast_level : planted bicluster mean effect size.
    row_overlap_ratio : overlap between consecutive row supports.
    col_overlap_ratio : overlap between consecutive column supports.
    background_noise_std : ambient noise level.
    row_effect_std : standard deviation of row effects, relative to contrast_level.
    col_effect_std : standard deviation of column effects, relative to contrast_level.
    within_block_noise_std : cell-level noise inside biclusters, relative to contrast_level.
    cell_signal_density : fraction of cells inside each bicluster receiving signal.
                          Default 1.0 gives full rectangular biclusters.

    Returns
    -------
    result : dict
        Contains shuffled data, unshuffled data, ground truth indicators,
        signal matrix, noise matrix, permutations, and metadata.
    """
    rng = np.random.default_rng(seed)

    contrast = _as_array(
        contrast_level, n_clusters, dtype=float, name="contrast_level"
    )

    row_indicators, row_supports = _make_supports(
        total_size=total_rows,
        n_clusters=n_clusters,
        size_fraction=row_fraction,
        overlap_ratio=row_overlap_ratio,
        rng=rng,
        placement=placement,
        size_jitter=size_jitter,
    )

    col_indicators, col_supports = _make_supports(
        total_size=total_cols,
        n_clusters=n_clusters,
        size_fraction=col_fraction,
        overlap_ratio=col_overlap_ratio,
        rng=rng,
        placement=placement,
        size_jitter=size_jitter,
    )

    noise = _sample_noise(
        rng,
        shape=(total_rows, total_cols),
        noise_std=background_noise_std,
        noise_distribution=noise_distribution,
        df=noise_df,
    ).astype(dtype)

    signal = np.zeros((total_rows, total_cols), dtype=dtype)

    cluster_params = []

    for k in range(n_clusters):
        rows = row_supports[k]
        cols = col_supports[k]

        block_sign = rng.choice([-1.0, 1.0]) if signed_blocks else 1.0
        mu_k = block_sign * contrast[k]

        alpha = rng.normal(
            0.0,
            row_effect_std * abs(contrast[k]),
            size=len(rows),
        )

        beta = rng.normal(
            0.0,
            col_effect_std * abs(contrast[k]),
            size=len(cols),
        )

        block_noise = rng.normal(
            0.0,
            within_block_noise_std * abs(contrast[k]),
            size=(len(rows), len(cols)),
        )

        block = mu_k + alpha[:, None] + beta[None, :] + block_noise

        if not (0 < cell_signal_density <= 1):
            raise ValueError("cell_signal_density must be in (0, 1].")

        if cell_signal_density < 1.0:
            cell_mask = rng.random(size=block.shape) < cell_signal_density
            block = block * cell_mask

        signal[np.ix_(rows, cols)] += block

        cluster_params.append(
            {
                "cluster": k,
                "n_rows": len(rows),
                "n_cols": len(cols),
                "mu": mu_k,
                "contrast_level": contrast[k],
            }
        )

    data = signal + noise

    (
        shuffled_data,
        shuffled_row_indicators,
        shuffled_col_indicators,
        row_perm,
        col_perm,
    ) = _shuffle_outputs(data, row_indicators, col_indicators, rng)

    return {
        "shuffled_data": shuffled_data.astype(dtype),
        "data": data.astype(dtype),
        "signal_matrix": signal.astype(dtype),
        "noise_matrix": noise.astype(dtype),
        "true_row_indicators": row_indicators,
        "true_col_indicators": col_indicators,
        "shuffled_row_indicators": shuffled_row_indicators,
        "shuffled_col_indicators": shuffled_col_indicators,
        "row_perm": row_perm,
        "col_perm": col_perm,
        "row_supports": row_supports,
        "col_supports": col_supports,
        "cluster_params": cluster_params,
        "metadata": {
            "model": "additive",
            "n_clusters": n_clusters,
            "total_rows": total_rows,
            "total_cols": total_cols,
            "row_fraction": row_fraction,
            "col_fraction": col_fraction,
            "contrast_level": contrast_level,
            "row_overlap_ratio": row_overlap_ratio,
            "col_overlap_ratio": col_overlap_ratio,
            "background_noise_std": background_noise_std,
            "row_effect_std": row_effect_std,
            "col_effect_std": col_effect_std,
            "within_block_noise_std": within_block_noise_std,
            "size_jitter": size_jitter,
            "placement": placement,
            "noise_distribution": noise_distribution,
            "signed_blocks": signed_blocks,
            "cell_signal_density": cell_signal_density,
            "seed": seed,
        },
    }

In [7]:
def generate_multiplicative_biclusters(
    n_clusters=3,
    total_rows=1000,
    total_cols=100,
    row_fraction=0.1,
    col_fraction=0.1,
    contrast_level=3.0,
    row_overlap_ratio=0.15,
    col_overlap_ratio=0.15,
    background_noise_std=1.0,
    latent_variability=0.15,
    factor_background_std=0.0,
    size_jitter=0.0,
    placement="diagonal",
    noise_distribution="gaussian",
    noise_df=5,
    signed_rows=True,
    signed_factors=False,
    seed=42,
    dtype=np.float64,
):
    """
    Generate synthetic biclustering data using a multiplicative latent factor model.

    Model
    -----
    X = Lambda @ Z + E

    Main controlled factors
    -----------------------
    n_clusters : number of biclusters.
    row_fraction : row support size as fraction of total rows.
    col_fraction : column support size as fraction of total columns.
    contrast_level : expected planted product magnitude inside biclusters.
    row_overlap_ratio : overlap between consecutive row supports.
    col_overlap_ratio : overlap between consecutive column supports.
    background_noise_std : ambient noise level.
    latent_variability : variability of active Lambda and Z entries.
    factor_background_std : nonzero value gives weak global latent background.
                            Use 0.0 for clean sparse FABIA-style factors.

    Returns
    -------
    result : dict
        Contains shuffled data, unshuffled data, true factors, ground truth
        indicators, signal matrix, noise matrix, permutations, and metadata.
    """
    rng = np.random.default_rng(seed)

    contrast = _as_array(
        contrast_level, n_clusters, dtype=float, name="contrast_level"
    )

    if np.any(contrast <= 0):
        raise ValueError("contrast_level must be positive for multiplicative data.")

    row_indicators, row_supports = _make_supports(
        total_size=total_rows,
        n_clusters=n_clusters,
        size_fraction=row_fraction,
        overlap_ratio=row_overlap_ratio,
        rng=rng,
        placement=placement,
        size_jitter=size_jitter,
    )

    col_indicators, col_supports = _make_supports(
        total_size=total_cols,
        n_clusters=n_clusters,
        size_fraction=col_fraction,
        overlap_ratio=col_overlap_ratio,
        rng=rng,
        placement=placement,
        size_jitter=size_jitter,
    )

    Lambda = rng.normal(
        0.0,
        factor_background_std,
        size=(total_rows, n_clusters),
    ).astype(dtype)

    Z = rng.normal(
        0.0,
        factor_background_std,
        size=(n_clusters, total_cols),
    ).astype(dtype)

    cluster_params = []

    for k in range(n_clusters):
        rows = row_supports[k]
        cols = col_supports[k]

        # Split the desired product magnitude evenly across Lambda and Z.
        latent_mean = np.sqrt(contrast[k])

        row_values = latent_mean + rng.normal(
            0.0,
            latent_variability * latent_mean,
            size=len(rows),
        )

        col_values = latent_mean + rng.normal(
            0.0,
            latent_variability * latent_mean,
            size=len(cols),
        )

        if signed_rows:
            row_values *= rng.choice([-1.0, 1.0], size=len(rows))

        if signed_factors:
            factor_sign = rng.choice([-1.0, 1.0])
            row_values *= factor_sign

        Lambda[rows, k] = row_values
        Z[k, cols] = col_values

        cluster_params.append(
            {
                "cluster": k,
                "n_rows": len(rows),
                "n_cols": len(cols),
                "contrast_level": contrast[k],
                "latent_mean": latent_mean,
            }
        )

    signal = Lambda @ Z

    noise = _sample_noise(
        rng,
        shape=(total_rows, total_cols),
        noise_std=background_noise_std,
        noise_distribution=noise_distribution,
        df=noise_df,
    ).astype(dtype)

    data = signal + noise

    (
        shuffled_data,
        shuffled_row_indicators,
        shuffled_col_indicators,
        row_perm,
        col_perm,
    ) = _shuffle_outputs(data, row_indicators, col_indicators, rng)

    return {
        "shuffled_data": shuffled_data.astype(dtype),
        "data": data.astype(dtype),
        "signal_matrix": signal.astype(dtype),
        "noise_matrix": noise.astype(dtype),
        "Lambda": Lambda.astype(dtype),
        "Z": Z.astype(dtype),
        "true_row_indicators": row_indicators,
        "true_col_indicators": col_indicators,
        "shuffled_row_indicators": shuffled_row_indicators,
        "shuffled_col_indicators": shuffled_col_indicators,
        "row_perm": row_perm,
        "col_perm": col_perm,
        "row_supports": row_supports,
        "col_supports": col_supports,
        "cluster_params": cluster_params,
        "metadata": {
            "model": "multiplicative",
            "n_clusters": n_clusters,
            "total_rows": total_rows,
            "total_cols": total_cols,
            "row_fraction": row_fraction,
            "col_fraction": col_fraction,
            "contrast_level": contrast_level,
            "row_overlap_ratio": row_overlap_ratio,
            "col_overlap_ratio": col_overlap_ratio,
            "background_noise_std": background_noise_std,
            "latent_variability": latent_variability,
            "factor_background_std": factor_background_std,
            "size_jitter": size_jitter,
            "placement": placement,
            "noise_distribution": noise_distribution,
            "signed_rows": signed_rows,
            "signed_factors": signed_factors,
            "seed": seed,
        },
    }

In [8]:
additive_result = generate_additive_biclusters(
    n_clusters=5,
    total_rows=1000,
    total_cols=100,
    row_fraction=0.12,
    col_fraction=0.20,
    contrast_level=2.5,
    row_overlap_ratio=0.25,
    col_overlap_ratio=0.10,
    background_noise_std=1.0,
    placement="diagonal",
    seed=123,
)

X_add = additive_result["shuffled_data"]
true_rows_add = additive_result["shuffled_row_indicators"]
true_cols_add = additive_result["shuffled_col_indicators"]

In [9]:
multiplicative_result = generate_multiplicative_biclusters(
    n_clusters=5,
    total_rows=1000,
    total_cols=100,
    row_fraction=0.12,
    col_fraction=0.20,
    contrast_level=2.5,
    row_overlap_ratio=0.25,
    col_overlap_ratio=0.10,
    background_noise_std=1.0,
    latent_variability=0.2,
    factor_background_std=0.0,
    placement="diagonal",
    seed=123,
)

X_mult = multiplicative_result["shuffled_data"]
true_rows_mult = multiplicative_result["shuffled_row_indicators"]
true_cols_mult = multiplicative_result["shuffled_col_indicators"]

In [10]:
# ============================================================
# 1. Output folder
# ============================================================

BASE_DIR = Path("/Users/ilg/Desktop/year4/M4R/python_files/synthetic_datasets")

DATA_DIR = BASE_DIR / "data"
ROW_TRUTH_DIR = BASE_DIR / "true_row_indicators"
COL_TRUTH_DIR = BASE_DIR / "true_col_indicators"
SIGNAL_DIR = BASE_DIR / "signal_matrices"
METADATA_DIR = BASE_DIR / "metadata"

for folder in [DATA_DIR, ROW_TRUTH_DIR, COL_TRUTH_DIR, SIGNAL_DIR, METADATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [12]:
TOTAL_ROWS = 1000
TOTAL_COLS = 100

model_families = ["additive", "multiplicative"]

# Keep 3 levels of complexity.
n_clusters_list = [2, 3, 5]

# Keep contrast as the signal-strength/difficulty factor.
contrast_levels = [0.75, 1.5, 3.0]

# Same overlap level for rows and columns in the main design.
overlap_ratios = [0.00, 0.20, 0.40]

# Three random repeats.
seeds = [0, 1, 2]


# ------------------------------------------------------------
# Shape / sparsity settings
# ------------------------------------------------------------
# cell_fraction_per_bicluster = row_fraction * col_fraction
#
# For example:
# row_fraction = 0.05, col_fraction = 0.10
# gives 0.5% of all matrix cells per bicluster.
#
# Large and very large biclusters are restricted to smaller K
# because they cannot fit diagonally for K=5 when overlap is low.
# ------------------------------------------------------------

shape_settings = [
    {
        "shape_name": "very_sparse_local",
        "row_fraction": 0.03,
        "col_fraction": 0.08,
        "cell_fraction": 0.03 * 0.08,   # 0.24% per bicluster
        "allowed_n_clusters": [2, 3, 5],
    },
    {
        "shape_name": "sparse_local",
        "row_fraction": 0.05,
        "col_fraction": 0.10,
        "cell_fraction": 0.05 * 0.10,   # 0.50% per bicluster
        "allowed_n_clusters": [2, 3, 5],
    },
    {
        "shape_name": "medium_balanced",
        "row_fraction": 0.12,
        "col_fraction": 0.15,
        "cell_fraction": 0.12 * 0.15,   # 1.80% per bicluster
        "allowed_n_clusters": [2, 3, 5],
    },
    {
        "shape_name": "large_balanced",
        "row_fraction": 0.18,
        "col_fraction": 0.18,
        "cell_fraction": 0.18 * 0.18,   # 3.24% per bicluster
        "allowed_n_clusters": [2, 3, 5],
    },
    {
        "shape_name": "large_row_dominant",
        "row_fraction": 0.25,
        "col_fraction": 0.12,
        "cell_fraction": 0.25 * 0.12,   # 3.00% per bicluster
        "allowed_n_clusters": [2, 3],
    },
    {
        "shape_name": "large_col_dominant",
        "row_fraction": 0.12,
        "col_fraction": 0.25,
        "cell_fraction": 0.12 * 0.25,   # 3.00% per bicluster
        "allowed_n_clusters": [2, 3],
    },
    {
        "shape_name": "huge_balanced",
        "row_fraction": 0.30,
        "col_fraction": 0.30,
        "cell_fraction": 0.30 * 0.30,   # 9.00% per bicluster
        "allowed_n_clusters": [2, 3],
    },
    {
        "shape_name": "very_huge_balanced",
        "row_fraction": 0.40,
        "col_fraction": 0.40,
        "cell_fraction": 0.40 * 0.40,   # 20.25% per bicluster
        "allowed_n_clusters": [2],
    },
]


# Fixed settings for the main synthetic library.
BACKGROUND_NOISE_STD = 1.0
PLACEMENT = "diagonal"
SIZE_JITTER = 0.0
NOISE_DISTRIBUTION = "gaussian"

In [13]:
# ============================================================
# 3. Helper functions
# ============================================================

def float_to_name(x):
    """
    Convert float to filename-safe string.
    Example: 0.75 -> '0p75'
    """
    return f"{x:.2f}".replace(".", "p")


def save_array_csv(array, path, fmt="%.6g"):
    """
    Save a numpy array as CSV.
    """
    np.savetxt(path, array, delimiter=",", fmt=fmt)


def save_dataset_outputs(result, file_stem, metadata):
    """
    Save generated dataset, ground-truth indicators, signal matrix, 
    and metadata.
    """
    data_path = DATA_DIR / f"{file_stem}.csv"
    row_truth_path = ROW_TRUTH_DIR / f"{file_stem}_rows.csv"
    col_truth_path = COL_TRUTH_DIR / f"{file_stem}_cols.csv"
    signal_path = SIGNAL_DIR / f"{file_stem}_signal.csv"
    metadata_path = METADATA_DIR / f"{file_stem}.json"

    save_array_csv(result["shuffled_data"], data_path)

    if "shuffled_row_indicators" in result:
        save_array_csv(result["shuffled_row_indicators"], row_truth_path, fmt="%d")

    if "shuffled_col_indicators" in result:
        save_array_csv(result["shuffled_col_indicators"], col_truth_path, fmt="%d")

    if "signal_matrix" in result:
        save_array_csv(result["signal_matrix"], signal_path)

    metadata = metadata.copy()
    metadata["data_file"] = str(data_path)
    metadata["row_truth_file"] = str(row_truth_path)
    metadata["col_truth_file"] = str(col_truth_path)
    metadata["signal_file"] = str(signal_path)
    metadata["metadata_file"] = str(metadata_path)

    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=4)

    return metadata

In [14]:
def diagonal_supports_fit(fraction, n_clusters, overlap_ratio):
    """
    Checks approximately whether diagonal supports of equal size can fit.

    Required fraction is:
        fraction * (n_clusters - (n_clusters - 1) * overlap_ratio)

    This must be <= 1.
    """
    required_fraction = fraction * (n_clusters - (n_clusters - 1) * overlap_ratio)
    return required_fraction <= 1.0

In [15]:
def generate_null_dataset(
    null_type,
    total_rows=1000,
    total_cols=100,
    background_noise_std=1.0,
    seed=0,
    rank=3,
    global_strength=1.5,
):
    """
    Generate null/control datasets with no planted local biclusters.

    null_type:
        'gaussian_noise'
        'student_t_noise'
        'global_low_rank'
    """
    rng = np.random.default_rng(seed)

    if null_type == "gaussian_noise":
        data = rng.normal(
            0.0,
            background_noise_std,
            size=(total_rows, total_cols),
        )

    elif null_type == "student_t_noise":
        df = 5
        raw = rng.standard_t(df=df, size=(total_rows, total_cols))
        data = raw * background_noise_std / np.sqrt(df / (df - 2))

    elif null_type == "global_low_rank":
        U = rng.normal(0.0, 1.0, size=(total_rows, rank))
        V = rng.normal(0.0, 1.0, size=(rank, total_cols))

        low_rank_signal = global_strength * (U @ V) / np.sqrt(rank)
        noise = rng.normal(
            0.0,
            background_noise_std,
            size=(total_rows, total_cols),
        )

        data = low_rank_signal + noise

    else:
        raise ValueError(
            "null_type must be 'gaussian_noise', 'student_t_noise', or 'global_low_rank'."
        )

    row_perm = rng.permutation(total_rows)
    col_perm = rng.permutation(total_cols)

    shuffled_data = data[row_perm][:, col_perm]

    return {
        "shuffled_data": shuffled_data,
        "data": data,
        "row_perm": row_perm,
        "col_perm": col_perm,
    }


In [16]:
all_metadata = []
dataset_counter = 0
skipped_configs = []

for model_family in model_families:
    for n_clusters in n_clusters_list:
        for contrast_level in contrast_levels:
            for overlap_ratio in overlap_ratios:
                for shape in shape_settings:

                    shape_name = shape["shape_name"]
                    row_fraction = shape["row_fraction"]
                    col_fraction = shape["col_fraction"]

                    # Check allowed K first
                    if n_clusters not in shape["allowed_n_clusters"]:
                        skipped_configs.append({
                            "reason": "n_clusters_not_allowed",
                            "model_family": model_family,
                            "n_clusters": n_clusters,
                            "contrast_level": contrast_level,
                            "overlap_ratio": overlap_ratio,
                            "shape_name": shape_name,
                            "row_fraction": row_fraction,
                            "col_fraction": col_fraction,
                        })
                        continue

                    # Then check geometric fit
                    row_fits = diagonal_supports_fit(
                        row_fraction, n_clusters, overlap_ratio
                    )
                    col_fits = diagonal_supports_fit(
                        col_fraction, n_clusters, overlap_ratio
                    )

                    if not (row_fits and col_fits):
                        print(
                            f"Skipping {shape_name}, K={n_clusters}, "
                            f"overlap={overlap_ratio}: does not fit diagonally."
                        )

                        skipped_configs.append({
                            "reason": "does_not_fit_diagonally",
                            "model_family": model_family,
                            "n_clusters": n_clusters,
                            "contrast_level": contrast_level,
                            "overlap_ratio": overlap_ratio,
                            "shape_name": shape_name,
                            "row_fraction": row_fraction,
                            "col_fraction": col_fraction,
                        })
                        continue

                    for seed in seeds:

                        dataset_counter += 1

                        contrast_name = float_to_name(contrast_level)
                        overlap_name = float_to_name(overlap_ratio)

                        file_stem = (
                            f"synth_{dataset_counter:04d}_"
                            f"{model_family}_"
                            f"K{n_clusters}_"
                            f"C{contrast_name}_"
                            f"O{overlap_name}_"
                            f"{shape_name}_"
                            f"seed{seed}"
                        )

                        print(f"Generating {file_stem}")

                        if model_family == "additive":
                            result = generate_additive_biclusters(
                                n_clusters=n_clusters,
                                total_rows=TOTAL_ROWS,
                                total_cols=TOTAL_COLS,
                                row_fraction=row_fraction,
                                col_fraction=col_fraction,
                                contrast_level=contrast_level,
                                row_overlap_ratio=overlap_ratio,
                                col_overlap_ratio=overlap_ratio,
                                background_noise_std=BACKGROUND_NOISE_STD,
                                placement=PLACEMENT,
                                size_jitter=SIZE_JITTER,
                                noise_distribution=NOISE_DISTRIBUTION,
                                seed=seed,
                            )

                        elif model_family == "multiplicative":
                            result = generate_multiplicative_biclusters(
                                n_clusters=n_clusters,
                                total_rows=TOTAL_ROWS,
                                total_cols=TOTAL_COLS,
                                row_fraction=row_fraction,
                                col_fraction=col_fraction,
                                contrast_level=contrast_level,
                                row_overlap_ratio=overlap_ratio,
                                col_overlap_ratio=overlap_ratio,
                                background_noise_std=BACKGROUND_NOISE_STD,
                                placement=PLACEMENT,
                                size_jitter=SIZE_JITTER,
                                noise_distribution=NOISE_DISTRIBUTION,
                                seed=seed,
                            )

                        else:
                            raise ValueError("Unknown model family.")

                        metadata = {
                            "dataset_id": dataset_counter,
                            "dataset_type": "structured",
                            "model_family": model_family,
                            "n_clusters": n_clusters,
                            "total_rows": TOTAL_ROWS,
                            "total_cols": TOTAL_COLS,
                            "contrast_level": contrast_level,
                            "row_overlap_ratio": overlap_ratio,
                            "col_overlap_ratio": overlap_ratio,
                            "shape_name": shape_name,
                            "row_fraction": row_fraction,
                            "col_fraction": col_fraction,
                            "cell_fraction": shape["cell_fraction"],
                            "background_noise_std": BACKGROUND_NOISE_STD,
                            "placement": PLACEMENT,
                            "size_jitter": SIZE_JITTER,
                            "noise_distribution": NOISE_DISTRIBUTION,
                            "seed": seed,
                            "has_planted_biclusters": True,
                            "file_stem": file_stem,
                        }

                        saved_metadata = save_dataset_outputs(
                            result=result,
                            file_stem=file_stem,
                            metadata=metadata,
                        )

                        all_metadata.append(saved_metadata)

Generating synth_0001_additive_K2_C0p75_O0p00_very_sparse_local_seed0
Generating synth_0002_additive_K2_C0p75_O0p00_very_sparse_local_seed1
Generating synth_0003_additive_K2_C0p75_O0p00_very_sparse_local_seed2
Generating synth_0004_additive_K2_C0p75_O0p00_sparse_local_seed0
Generating synth_0005_additive_K2_C0p75_O0p00_sparse_local_seed1
Generating synth_0006_additive_K2_C0p75_O0p00_sparse_local_seed2
Generating synth_0007_additive_K2_C0p75_O0p00_medium_balanced_seed0
Generating synth_0008_additive_K2_C0p75_O0p00_medium_balanced_seed1
Generating synth_0009_additive_K2_C0p75_O0p00_medium_balanced_seed2
Generating synth_0010_additive_K2_C0p75_O0p00_large_balanced_seed0
Generating synth_0011_additive_K2_C0p75_O0p00_large_balanced_seed1
Generating synth_0012_additive_K2_C0p75_O0p00_large_balanced_seed2
Generating synth_0013_additive_K2_C0p75_O0p00_large_row_dominant_seed0
Generating synth_0014_additive_K2_C0p75_O0p00_large_row_dominant_seed1
Generating synth_0015_additive_K2_C0p75_O0p00_la

In [18]:
# ============================================================
# 5. Generate null/control datasets
# ============================================================

null_types = ["gaussian_noise", "student_t_noise", "global_low_rank"]
null_seeds = [0, 1, 2, 3, 4, 5]

for null_type in null_types:
    for seed in null_seeds:

        dataset_counter += 1

        file_stem = (
            f"synth_{dataset_counter:04d}_"
            f"null_"
            f"{null_type}_"
            f"seed{seed}"
        )

        print(f"Generating {file_stem}")

        result = generate_null_dataset(
            null_type=null_type,
            total_rows=TOTAL_ROWS,
            total_cols=TOTAL_COLS,
            background_noise_std=BACKGROUND_NOISE_STD,
            seed=seed,
        )

        metadata = {
            "dataset_id": dataset_counter,
            "dataset_type": "null",
            "model_family": "null",
            "null_type": null_type,
            "n_clusters": 0,
            "total_rows": TOTAL_ROWS,
            "total_cols": TOTAL_COLS,
            "contrast_level": None,
            "row_overlap_ratio": None,
            "col_overlap_ratio": None,
            "shape_name": None,
            "row_fraction": None,
            "col_fraction": None,
            "background_noise_std": BACKGROUND_NOISE_STD,
            "placement": None,
            "size_jitter": None,
            "noise_distribution": null_type,
            "seed": seed,
            "has_planted_biclusters": False,
            "file_stem": file_stem,
        }

        # For null datasets, there are no row/column truth indicators.
        data_path = DATA_DIR / f"{file_stem}.csv"
        metadata_path = METADATA_DIR / f"{file_stem}.json"

        save_array_csv(result["shuffled_data"], data_path)

        metadata["data_file"] = str(data_path)
        metadata["row_truth_file"] = None
        metadata["col_truth_file"] = None
        metadata["signal_file"] = None
        metadata["metadata_file"] = str(metadata_path)

        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=4)

        all_metadata.append(metadata)


Generating synth_1027_null_gaussian_noise_seed0
Generating synth_1028_null_gaussian_noise_seed1
Generating synth_1029_null_gaussian_noise_seed2
Generating synth_1030_null_gaussian_noise_seed3
Generating synth_1031_null_gaussian_noise_seed4
Generating synth_1032_null_gaussian_noise_seed5
Generating synth_1033_null_student_t_noise_seed0
Generating synth_1034_null_student_t_noise_seed1
Generating synth_1035_null_student_t_noise_seed2
Generating synth_1036_null_student_t_noise_seed3
Generating synth_1037_null_student_t_noise_seed4
Generating synth_1038_null_student_t_noise_seed5
Generating synth_1039_null_global_low_rank_seed0
Generating synth_1040_null_global_low_rank_seed1
Generating synth_1041_null_global_low_rank_seed2
Generating synth_1042_null_global_low_rank_seed3
Generating synth_1043_null_global_low_rank_seed4
Generating synth_1044_null_global_low_rank_seed5


In [19]:
# ============================================================
# 6. Save master metadata table
# ============================================================

metadata_df = pd.DataFrame(all_metadata)
metadata_csv_path = BASE_DIR / "synthetic_dataset_metadata.csv"
metadata_df.to_csv(metadata_csv_path, index=False)

print("\nFinished generating synthetic datasets.")
print(f"Total datasets generated: {dataset_counter}")
print(f"Metadata saved to: {metadata_csv_path}")
print(f"Data saved in: {DATA_DIR}")


Finished generating synthetic datasets.
Total datasets generated: 1044
Metadata saved to: /Users/ilg/Desktop/year4/M4R/python_files/synthetic_datasets/synthetic_dataset_metadata.csv
Data saved in: /Users/ilg/Desktop/year4/M4R/python_files/synthetic_datasets/data
